# YOLO-World-Dreamer · CPU Colab

在 DreamerV3 RSSM 世界模型基座上嫁接 **YOLO26 端到端候选 + YOLOv10 双头分配 + YOLOE 文本点乘**,做**语言目标条件**规划的 Crafter 智能体。

**两条线**(`knowledge/yoloworld.md`):
- **世界模型线**:真实序列(长 16/32)自监督 —— 重建 + 奖励 + 终止 + **成就头 ψ** BCE + KL。
- **行为双头线**:`256` 候选动作序列**小头**(采集动作时只跑它,端到端无搜索)+ **top-M rollout 老师**(把候选喂进世界模型想象、算回报、给小头当监督)。

**任务用语言描述**:22 个 Crafter 成就各一句指令 → 冻结 MiniLM 编码 `E∈R^{22×384}`;小头每条候选嵌入 `e^k` 与任务编码 `g` **点乘**直接选最优序列(YOLOE)。逐 env 不同目标,最大化任务多样性。

> 本 notebook 用 **CPU** 跑通正确性 + 小规模出信号。真正训到 Crafter 分数需同份代码挂 **GPU**(末尾说明)。

## 1. 环境(CPU)

若在 Colab 新建会话,先 clone 仓库并 `cd` 进去;已在仓库根目录则跳过 clone。

In [ ]:
import os
if not os.path.exists('net/yoloworld'):
    # 在 Colab 全新会话里取消注释并填入你的仓库地址:
    # !git clone <YOUR_REPO_URL> repo && cd repo
    raise SystemExit('请在仓库根目录运行(应能看到 net/yoloworld/)')
!pip -q install crafter transformers torch numpy

## 2. 结构自检(集成测试,CPU 秒级)

前向 / 反向 / 行为线不回梯度到世界模型 / **slot 多样性**(反候选坍缩)。

In [ ]:
!python -m pytest tests/integration/test_yoloworld_build.py -q

## 3. CPU 冒烟训练(tiny)

短跑观察信号是否朝正确方向走:`wm`/`img`/`ach` 下降、`align` 下降(YOLOE 对齐)、`cls` 离开 `log(M+ε)` 上限(双头选择在学老师)、`ach/ep` 上升、slot 不坍缩。

In [ ]:
!python -m train.crafter.train_yoloworld \
    --size tiny --device cpu --total-steps 3000 --prefill 500 \
    --n-envs 8 --batch-size 16 --seq-len 16 --n-start 32 --her-ratio 0.5 \
    --task-encoder minilm --updates-per 1 --log-interval 20 \
    --save-interval 100000 --run-dir runs/colab_yw

## 4. 切到 GPU 训到分数

同份代码,换 `--size crafter --device cuda`(deter=512 / 离散 32×32 / units=512 / K=256 / H=16,~1.7×10⁷ 参数,DreamerV3 已验证能学会 Crafter)。

```bash
python -m train.crafter.train_yoloworld \
    --size crafter --device cuda --total-steps 1000000 \
    --n-envs 16 --batch-size 32 --seq-len 32 --updates-per 1 --her-ratio 0.5
```

**利用率旋钮**:
- GPU:`--batch-size`/`--n-envs` 加大喂饱;rollout 老师已矢量化成 `[n_start·(M+ε)]` 单批沿 H 一次 scan;`--n-start 0` 用全部 B·L 起点最大化批维;TF32+cudnn.benchmark 训练入口自动开。
- CPU:`set_num_threads = 核数`(自动);`--n-start` 调小控候选 rollout 算力。
- `--her-ratio` 控 HER 事后重标比例(目标条件稀疏信号变稠密);`--task-encoder mock` 跳过 MiniLM 下载(无语义外推,仅冒烟)。

设计与数学推导见 [`knowledge/yoloworld.md`](knowledge/yoloworld.md)。